# B1 window Transformer-only

**Later illustrated companion; not the executed training notebook.** Historical metrics below are quoted from copied run records, never represented as notebook outputs.

A budget-paused run is not converged; the target was four epochs with checkpoint selection and train-only calibration. The v7 source also defines gated depth expansion and an upper FFN option, but the recorded configs for these exports keep encoder_depth=6 and expand_ffn=false; those unlaunched options have no reported score. Original checkpoint bytes were downloaded and verified in the local archive, but are omitted from GitHub.

## Task and architecture

B1 6-encoder/6-decoder original Phase 1 backbone; 512 tokens per chunk, 16 ordered chunks per case, overlap 100, tail sampling. Two document Transformer blocks without document BiGRU. Token representation uses a learned mixture of mean, attention and last content token; position enters before document aggregation. The final learned attention pools document states for two-class logits.

**Adaptation:** Staged B1 fine-tuning; document head mode `transformer_only`. Original B1 study notebook lineage `code/v1.ipynb` through `code/v7.ipynb`; latest source snapshot `code/source/b1_study/`. Saved configs and terminal/run evidence are in `evidence/`.

Data path: official train → training-only fit/selection → official labeled dev report → 1,517-label-free test predictions. The original `code/` and `evidence/` directories contain the audit trail.

## Recorded result

Version 7 run state **paused**; 7292 optimizer updates and 1 complete epochs, selected step 7292. Calibrated official 994-case dev macro-F1 **0.674622** at threshold 0.392977; default 0.5 dev macro-F1 0.688851. Selected checkpoint SHA-256 `f122c53d14d36ffded73bff9bb7f1845669c3c143b2448ebde3efaa128a2aeb0`; verified 1,517-ID ZIP SHA-256 `de0b4be13dc70a4812211d001a51eb3557d27e5391b8e08bad13397c86384803`.

Local official dev, calibration CV, and Codabench scores are different measurements. A historical score is not an output of this notebook.

## Source excerpt: `model.py`

The complete source is in `code/`. This excerpt illustrates the exact preserved implementation; it is not a training replay.

In [ ]:
import copy
from dataclasses import replace

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

from phase3_rr.models import load_backbone
from phase4_cjpe.models import EvidenceCaseModel


class GatedEncoderBlock(nn.Module):
    def __init__(self, block):
        super().__init__()
        self.block = copy.deepcopy(block)
        self.attention_gate = nn.Parameter(torch.zeros(()))
        self.ffn_gate = nn.Parameter(torch.zeros(()))

    def forward(self, x, valid_mask):
        x = x + self.attention_gate * self.block.attn(self.block.attn_norm(x), valid_mask)
        return x + self.ffn_gate * self.block.ffn(self.block.ffn_norm(x))


def grow_encoder(backbone):
    if len(backbone.encoder) != 6:
        raise ValueError('Depth expansion requires the original six encoder blocks')
    layers, mapping = [], []
    for i, block in enumerate(backbone.encoder):
        layers.append(block)
        mapping.append({'new_index': len(layers) - 1, 'original_index': i, 'gated_copy': False})
        if i in (2, 5):
            layers.append(GatedEncoderBlock(block))
            mapping.append({'new_index': len(layers) - 1, 'original_index': i, 'gated_copy': True})
    backbone.encoder = nn.ModuleList(layers)
    backbone.cfg = replace(backbone.cfg, n_encoder_layers=8)
    return mapping


def expand_upper_ffn(backbone, new_inner=3072):
    names = []
    targets = [('encoder', i, b) for i, b in enumerate(backbone.encoder)][-2:]
    targets += [('decoder', i, b) for i, b in enumerate(backbone.decoder)][-2:]
    for family, index, block in targets:
        block = block.block if isinstance(block, GatedEncoderBlock) else block
        ffn = block.ffn
        old, width = ffn.w1.weight.shape
        if new_inner <= old:
            raise ValueError('FFN expansion must increase inner width')
        device, dtype = ffn.w1.weight.device, ffn.w1.weight.dtype
        first = nn.Linear(width, new_inner, bias=False, device=device, dtype=dtype)
        third = nn.Linear(width, new_inner, bias=False, device=device, dtype=dtype)
        second = nn.Linear(new_inner, width, bias=False, device=device, dtype=dtype)
        with torch.no_grad():
            first.weight[:old].copy_(ffn.w1.weight)
            third.weight[:old].copy_(ffn.w3.weight)
            second.weight.zero_()
            second.weight[:, :old].copy_(ffn.w2.weight)
        ffn.w1, ffn.w2, ffn.w3 = first, second, third
        names.append(f'{family}.{index}.ffn:{old}->{new_inner}')
    return names


class DocumentBlock(nn.Module):
    def __init__(self, width=256, heads=4, inner=768, dropout=.1):
        super().__init__()
        self.heads, self.dropout = heads, dropout
        self.attention_norm, self.ffn_norm = nn.LayerNorm(width), nn.LayerNorm(width)
        self.qkv = nn.Linear(width, 3 * width)
        self.output = nn.Linear(width, width)
        self.up, self.gate = nn.Linear(width, inner), nn.Linear(width, inner)
        self.down = nn.Linear(inner, width)

    def forward(self, x):
        b, length, width = x.shape
        q, k, v = [a.reshape(b, length, self.heads, width // self.heads).transpose(1, 2)
                   for a in self.qkv(self.attention_norm(x)).chunk(3, -1)]
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.)
        x = x + F.dropout(self.output(out.transpose(1, 2).reshape(b, length, width)), self.dropout, self.training)
        z = self.ffn_norm(x)
        return x + F.dropout(self.down(F.silu(self.gate(z)) * self.up(z)), self.dropout, self.training)


class B1Study(EvidenceCaseModel):
    def __init__(self, backbone, config):

## Reproduction boundary

The package has code, configs, selected records, and prediction ZIP. Original model weights and official data are retained in the local archives, not duplicated here. Paths in the original configs must be remapped before an authorized rerun.